# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import login, list_repo_files, hf_hub_download
import pandas as pd
import numpy as np
import os

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
march_files = [f for f in files if "fact_content_daily_performance/month=2026-03" in f]
local_paths = [
    hf_hub_download(repo_id="FlyRank/internship-warehouse", filename=f, repo_type="dataset")
    for f in march_files
]
df_raw = pd.concat([pd.read_parquet(p) for p in local_paths], ignore_index=True)
print("Raw rows:", df_raw.shape)

df = df_raw[df_raw['gsc_data_available'] == True].copy()
print(f"GSC-available rows: {len(df)} of {len(df_raw)}")

agg = df.groupby(['client_hash_id', 'content_hash_id']).agg(
    days_with_data=('report_date', 'nunique'),
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    avg_position=('gsc_avg_position', 'mean')
).reset_index()

agg = agg[agg['total_impressions'] > 0].copy()
agg['ctr'] = agg['total_clicks'] / agg['total_impressions']

print("Content-level rows:", agg.shape)
print(agg[['total_impressions', 'total_clicks', 'avg_position', 'ctr']].describe())

Raw rows: (9841378, 30)
GSC-available rows: 3611061 of 9841378
Content-level rows: (176738, 7)
       total_impressions   total_clicks   avg_position            ctr
count      176738.000000  176738.000000  176738.000000  176738.000000
mean         1587.986675       4.650002      15.999277       0.004594
std          5431.337724      26.722649      17.686260       0.037760
min             1.000000       0.000000       0.000000       0.000000
25%            20.000000       0.000000       5.001970       0.000000
50%           173.000000       0.000000       8.505296       0.000000
75%          1039.000000       2.000000      20.369190       0.002158
max        617124.000000    5668.000000     309.000000       1.000000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
pos_bins = [0, 3, 10, 20, 50, 1000]
pos_labels = ['1-3', '4-10', '11-20', '21-50', '51+']
agg['position_bucket'] = pd.cut(agg['avg_position'], bins=pos_bins, labels=pos_labels)

signal1_table = agg.groupby('position_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks_sum=('total_clicks', 'sum'),
    impressions_sum=('total_impressions', 'sum')
)
signal1_table['weighted_ctr'] = signal1_table['clicks_sum'] / signal1_table['impressions_sum']
print(signal1_table[['n', 'weighted_ctr']])

                     n  weighted_ctr
position_bucket                     
1-3              16144      0.004063
4-10             81988      0.003239
11-20            32203      0.003052
21-50            33288      0.001366
51+              11681      0.000392


Signal 1 — CTR vs. position. Claim: pages ranking better get clicked more. Verdict: CONFIRMED. Weighted CTR falls from 0.41% (position 1–3, n=16,144) to 0.04% (position 51+, n=11,681) — a roughly 10× decline across the position range, and the pattern is monotonic across all five buckets. Every bucket clears the ~50-row floor. This is the signal behind FlyRank's CTR-fix flag: a page's expected CTR is set by its position, so a page earning far less than its bucket's norm is a candidate for a title/snippet fix rather than a rankings problem.

In [3]:
vol_bins = [0, 50, 200, 1000, 5000, 10_000_000]
vol_labels = ['0-50', '51-200', '201-1000', '1001-5000', '5000+']
agg['volume_bucket'] = pd.cut(agg['total_impressions'], bins=vol_bins, labels=vol_labels)

signal2_table = agg.groupby('volume_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks_sum=('total_clicks', 'sum'),
    impressions_sum=('total_impressions', 'sum'),
    avg_position=('avg_position', 'mean')
)
signal2_table['weighted_ctr'] = signal2_table['clicks_sum'] / signal2_table['impressions_sum']
print(signal2_table[['n', 'weighted_ctr', 'avg_position']])

                   n  weighted_ctr  avg_position
volume_bucket                                   
0-50           61015      0.004544     16.996009
51-200         31014      0.002397     22.071025
201-1000       39674      0.002428     15.376284
1001-5000      31745      0.003046     10.852553
5000+          13290      0.002936     11.407448


Signal 2 — Volume. Claim: higher-impression pages are worth prioritizing first because a CTR fix there returns more clicks per unit of editor effort. Verdict: MIXED. Weighted CTR does not move cleanly with volume (0.45% → 0.24% → 0.24% → 0.30% → 0.29% across the five buckets — no monotonic trend, all n well above 50). Average position does broadly improve with volume (17.0 → 22.1 → 15.4 → 10.9 → 11.4), meaning high-volume pages tend to already rank better — so volume is correlated with existing performance, not an independent predictor of a CTR gap. I still use volume in the rule below, but only as a multiplier on identified CTR gaps (same % gap = more absolute clicks recovered at higher volume) — not as a standalone confirmed signal.

Rule : A page is worth an editor's attention first if it already gets meaningful search visibility this month (1,000+ impressions) but its actual click-through rate is less than half of what pages at its own position typically earn — visible but being skipped, which usually means a fixable snippet/title problem, not a rankings problem.

Reason codes: CTR_GAP_HIGH_VISIBILITY, LOW_VISIBILITY_MONITOR, ON_TARGET

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
bucket_totals = agg.groupby('position_bucket', observed=True).agg(
    clicks_sum=('total_clicks', 'sum'),
    impressions_sum=('total_impressions', 'sum')
)
bucket_totals['expected_ctr'] = bucket_totals['clicks_sum'] / bucket_totals['impressions_sum']

agg['expected_ctr'] = agg['position_bucket'].map(bucket_totals['expected_ctr']).astype(float)
agg['ctr'] = agg['ctr'].astype(float)
agg['ctr_gap'] = agg['expected_ctr'] - agg['ctr']

print(agg[['position_bucket', 'ctr', 'expected_ctr', 'ctr_gap']].head(10))

  position_bucket       ctr  expected_ctr   ctr_gap
0            4-10  0.000000      0.003239  0.003239
1           11-20  0.006042      0.003052 -0.002990
2            4-10  0.000000      0.003239  0.003239
3            4-10  0.000000      0.003239  0.003239
4           11-20  0.000000      0.003052  0.003052
5           11-20  0.000000      0.003052  0.003052
6            4-10  0.025641      0.003239 -0.022402
7           11-20  0.000000      0.003052  0.003052
8            4-10  0.000000      0.003239  0.003239
9            4-10  0.000000      0.003239  0.003239


In [5]:
agg['ctr_ratio'] = agg['ctr'] / agg['expected_ctr'].replace(0, np.nan)

def score_row(row):
    if row['total_impressions'] >= 1000 and pd.notnull(row['ctr_ratio']) and row['ctr_ratio'] < 0.5:
        score = row['ctr_gap'] * row['total_impressions']
        reason = 'CTR_GAP_HIGH_VISIBILITY'
        action = 'fix_snippet'
    elif row['total_impressions'] < 200:
        score = 0
        reason = 'LOW_VISIBILITY_MONITOR'
        action = 'monitor'
    else:
        score = 0
        reason = 'ON_TARGET'
        action = 'monitor'
    return pd.Series([score, reason, action])

agg[['baseline_score', 'reason_code', 'action_label']] = agg.apply(score_row, axis=1)
print(agg['reason_code'].value_counts())

reason_code
LOW_VISIBILITY_MONITOR     91905
ON_TARGET                  66931
CTR_GAP_HIGH_VISIBILITY    17902
Name: count, dtype: int64


In [6]:
queue = agg.sort_values('baseline_score', ascending=False)[
    ['client_hash_id', 'content_hash_id', 'baseline_score', 'reason_code', 'action_label',
     'total_impressions', 'total_clicks', 'avg_position', 'ctr', 'expected_ctr', 'ctr_gap', 'ctr_ratio']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue)} rows. Flagged (score > 0): {(queue['baseline_score'] > 0).sum()}")

Wrote 176738 rows. Flagged (score > 0): 17902


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
print(queue.head(20).to_string(index=False))

         client_hash_id          content_hash_id  baseline_score             reason_code action_label  total_impressions  total_clicks  avg_position      ctr  expected_ctr  ctr_gap  ctr_ratio
client_23a62021009f63c4 content_44f34c0a90047651      663.966421 CTR_GAP_HIGH_VISIBILITY  fix_snippet             212404            24      7.346909 0.000113      0.003239 0.003126   0.034885
client_e547b89c05043229 content_8d7d99f109e19aa2      537.712743 CTR_GAP_HIGH_VISIBILITY  fix_snippet             203497           289      2.563756 0.001420      0.004063 0.002642   0.349577
client_73cda7b4e4f265ea content_8e1334d6356668e3      436.206735 CTR_GAP_HIGH_VISIBILITY  fix_snippet             134984             1      4.545582 0.000007      0.003239 0.003232   0.002287
client_62f4a7e64f5e0096 content_34a70fea29d15f24      420.231717 CTR_GAP_HIGH_VISIBILITY  fix_snippet             143019            43      3.219473 0.000301      0.003239 0.002938   0.092826
client_73cda7b4e4f265ea content_fec55986

**Top-20 review**

1. `content_44f34c0a90047651` — **fix_snippet** — 212k impressions, avg position 7.3, but only 24 clicks (CTR 0.011% vs. 0.32% expected). Would be wrong if this is a duplicate/redirect page whose impressions are real but whose title genuinely doesn't match the query intent — a content rewrite, not a snippet tweak, would be needed.
2. `content_8d7d99f109e19aa2` — **fix_snippet** — 203k impressions, position 2.6, CTR 0.14% vs. 0.41% expected. Would be wrong if a competing internal page or a featured snippet from elsewhere is siphoning the clicks this page would normally get.
3. `content_8e1334d6356668e3` — **fix_snippet** — 135k impressions, position 4.5, only 1 click all month. Would be wrong if `gsc_clicks` here is a data artifact (e.g., a tracking gap) rather than a real snippet problem — 1 click on 135k impressions is extreme enough to double-check against GA4 before acting.
4. `content_34a70fea29d15f24` — **fix_snippet** — 143k impressions, position 3.2, CTR 0.03% vs. 0.32% expected. Would be wrong if the page recently changed its target query and impressions haven't caught up to a genuine intent mismatch yet.
5. `content_fec55986a1868d62` — **fix_snippet** — 124k impressions, position 9.4, 1 click. Same caution as #3 — near-zero clicks on six-figure impressions is worth a manual GSC screenshot check before assuming it's purely a title problem.
6. `content_7c6373141eae744a` — **fix_snippet** — 133k impressions, position 5.8, CTR 0.06% vs. 0.32% expected. Would be wrong if this page ranks for a broad/informational query where low CTR is normal regardless of snippet quality.
7. `content_f6116743b00afc2d` — **fix_snippet** — 108k impressions, position 9.5, CTR 0.01% vs. 0.32% expected. Same informational-intent caveat as #6.
8. `content_306bc78dff1eb683` — **fix_snippet** — 81k impressions, position 1.5, CTR 0.04% vs. 0.41% expected. Position 1.5 with near-zero CTR is the strongest kind of signal here — rank #1 pages should get high CTR almost by default, so this is a good candidate for the fix.
9. `content_acbcc847f8996314` — **fix_snippet** — 171k impressions, CTR ratio 0.47, just barely under the 0.5 cutoff. Would be wrong (i.e., this shouldn't really be flagged) if the cutoff is treated as a hard rule rather than a judgment call — this one's a borderline case, not a clear miss.
10. `content_cd3d932d4e1c8db0` — **fix_snippet** — 89k impressions, position 7.8, 4 clicks. Would be wrong if this client's tracking setup undercounts clicks systematically (worth checking `client_has_gsc` history for this client).
11. `content_9ef3d7516483e665` — **fix_snippet** — 89k impressions, position 2.5, CTR 0.10% vs. 0.41%. Reasonably strong candidate — good position, clear shortfall.
12. `content_046fc480045b88f5` — **fix_snippet** — 84k impressions, position 7.3, 6 clicks. Would be wrong if seasonal/trending impressions spiked this content item only briefly this month.
13. `content_f43118e089ecc69a` — **fix_snippet** — 139k impressions, CTR ratio 0.42 — closer to the cutoff, weaker signal than the top ones.
14. `content_9540d884af3e41fd` — **fix_snippet** — 82k impressions, position 7.8, 11 clicks, ratio 0.041 — strong gap.
15. `content_9c057b66c30a3abb` — **fix_snippet** — 84k impressions, 1 click — same 1-click caution as #3/#5.
16. `content_c46df0fa61530d86` — **fix_snippet** — 70k impressions, position 1.6, CTR 0.06% vs. 0.41% — position-1 near-zero CTR again, strong candidate like #8.
17. `content_425715547c6a3ea8` — **fix_snippet** — 72k impressions, position 6.4, 3 clicks — strong gap, low click count worth a sanity check.
18. `content_fc67675904376267` — **fix_snippet** — 60k impressions, position 2.3, CTR 0.03% vs. 0.41% — position-2 near-zero CTR, strong candidate.
19. `content_36fc1ee501ec072d` — **fix_snippet** — 73k impressions, position 6.5, ratio 0.068 — solid gap.
20. `content_e578ac84778da489` — **fix_snippet** — 118k impressions, ratio 0.427 — weaker, near the cutoff like #9 and #13.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rows #9, #13, and #20 (ctr_ratio between 0.42–0.47) sit right at the 0.5 cutoff — they're on the queue by a small margin and shouldn't be treated with the same confidence as the top handful. Rows #3, #5, and #15 have only 1 click against 100k+ impressions — that's extreme enough that I'd want to cross-check against GA4 sessions for the same content item before trusting it's a snippet problem and not a tracking gap.

A separate pattern worth naming: the top 20 is dominated by just five clients — client_62f4a7e64f5e0096 (6 rows), client_73cda7b4e4f265ea (5 rows), client_e547b89c05043229 (4 rows), client_a80fca3f171ed1de (2 rows), plus one from client_9958f0a7ae1df715 and one from client_23a62021009f63c4. That's a client-level concentration, not a spread across the portfolio — likely means a handful of clients have systematically low-CTR content (or a shared CMS/template issue), which is a useful finding on its own but means this queue should be read as "which clients need attention" almost as much as "which pages."

Leakage check: confirmed. baseline_score is built only from total_impressions, total_clicks, and avg_position, all summed/averaged strictly within March 2026 — no rows from later months, no future window. No FlyRank product flags (health score, existing quick-win tags, etc.) were used as inputs; this rule is meant to be compared against those in a later week, never built from them.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.